# Lesson 01 - AI ഏജന്റികൾക്ക് പരിചയം

**AI ഏജന്റുകൾക്ക് തുടക്കക്കാരുടെ** കോഴ്‌സിലെ ആദ്യ പാഠത്തിലേക്ക് സ്വാഗതം!

**AI ഏജന്റ്** എന്നത് ഒരു പ്രോഗ്രാമാണ്, ഇത് വലിയ ഭാഷാ മാതൃക (LLM) reasoning എഞ്ചിനായി ഉപയോഗിക്കുകയും യഥാർത്ഥ ലോകത്തിൽ *പ്രവർത്തനങ്ങൾ* നടക്കുകയും ചെയ്യുന്നു — APIകൾ വിളിക്കുക, ഡാറ്റാബേസുകൾ ക്വറി ചെയ്യുക, അല്ലെങ്കിൽ കോഡ് ഓടിക്കുക — ഉപയോഗിക്കുന്ന ആളിന്റെ വേണ്ടി ഒരു ലക്ഷ്യം പൂർത്തീകരിക്കാൻ.

ഈ നോട്ട്‌ബുക്കിൽ നിങ്ങൾ നിങ്ങളുടെ ആദ്യ ഏജന്റായ ഒരു **ട്രാവൽ ഏജന്റ്** ഉണ്ടാക്കും, അത് അവധിക്കാല വിനോദസഞ്ചാര ലക്ഷ്യങ്ങൾ сунушിക്കും. ഇത്രയും വഴി നിങ്ങൾ പഠിക്കുന്നതായിരിക്കും:

1. **Microsoft Agent Framework** ഉപയോഗിച്ച് Azure AI Foundry Agent Service-ലേക്ക് കണക്ട് ചെയ്യുന്നത്.
2. ഏജന്റിന് ഒരു **ഉപകരണം** നൽകുക — ഇത് കോളുചെയ്യാനാകുന്ന ഒരു പൊതു Python ഫംഗ്‌ഷൻ ആയിരിക്കും.
3. ഏജന്റ് പ്രവർത്തിപ്പിച്ച് അതിന്റെ പ്രതികരണങ്ങൾ പരിശോധിക്കുക.
4. ഏജന്റിന്റെ പ്രതികരണം ടോകൺ-ഓടെ ടോകൺ ആയി സ്റ്റ്രീം ചെയ്യുക.


## സജ്ജീകരണം

ഈ നോട്ട്‌ബുക്ക് പ്രവർത്തിപ്പിക്കുന്നതിന് മുമ്പ്, നിങ്ങൾക്ക് ഉറപ്പാക്കണം:

1. **ഒരു Azure AI Foundry പ്രോജക്ട്** ഡിപ്പ്ലോയ്ചെയ്ത ചാറ്റ് മാതൃകയോടെ (ഉദാ., `gpt-4o-mini`).
2. **Azure CLI ഉപയോഗിച്ച് ലോഗിൻ ചെയ്തിട്ടുണ്ട്** — നിങ്ങളുടെ ടെർമിനലിൽ `az login` റൺ ചെയ്യുക.
3. **ആവശ്യമായ പരിസ്ഥിതി വ്യത്യസങ്ങൾ സജ്ജീകരിച്ചിട്ടുള്ളത്:**
   - `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Azure AI Foundry പ്രോജക്ടിന്റെ എൻഡ്‌പോയിന്റ്.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — നിങ്ങളുടെ ഡിപ്പ്ലോയ്ചെയ്‌ത് മാതൃകയുടെ പേര്.

ുകറ്റിയ സെൽ താഴെ നിങ്ങൾക്ക് ആവശ്യമുള്ള പൈത്തൺ പാക്കേജുകൾ ഇൻസ്റ്റാൾ ചെയ്യുന്നു.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## നിങ്ങളുടെ ആദ്യ ഏജന്റ് സൃഷ്ടിക്കൽ

ഒരു ഏജന്റിന് രണ്ടുതരം കാര്യങ്ങൾ ആവശ്യമാണ്:

- **നിർദ്ദേശങ്ങൾ** അതിന് *ആരാണ്* എന്നും *എങ്ങിനെ പെരുമാറണം* എന്നും പറയുന്നതായി (ഒരു സിസ്റ്റം പ്രോംപ്).
- **ഉപകരണങ്ങൾ** — ഏജന്റ് വിവരങ്ങൾ കൈപ്പറ്റാൻ അല്ലെങ്കിൽ പ്രവർത്തനങ്ങൾ നടത്താൻ വിളിക്കാൻ കഴിയുന്ന `@tool` ഉപയോഗിച്ച് അലങ്കരിച്ച പൈതൺ ഫങ്ഷനുകൾ.

താഴെ, ജനപ്രിയമായ അവധിക്കാല ടൂർസ്ഥലങ്ങളുടെ പട്ടിക നൽകുന്ന ഒരു ലളിതമായ ഉപകരണം നിർവചിക്കുന്നു. ഉപയോക്താവ് യാത്ര ശിപാർശകൾ ചോദിക്കുമ്പോൾ ഏജന്റ് ഈ ഉപകരണം ഉപയോഗിക്കും.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

മികവുറ്റ അനുഭവത്തിനായി നിങ്ങൾക്ക് ഏജന്റിന്റെ പ്രതികരണം **stream** ചെയ്യാൻ കഴിയും. മുഴുവൻ മറുപടി ആവശ്യമാവുന്നതിന് കാത്തിരിക്കാതെ, ഏജന്റ് ഉളവാകുന്നത് പോലെ എഴുത്ത് കഷണങ്ങൾ നൽകുന്നു. നിങ്ങൾക്ക് യഥാർത്ഥ സമയത്ത് ഔട്ട്പുട്ട് പ്രദർശിപ്പിക്കേണ്ട ഫ്ലാറ്റ്ഫോമുകളിൽ ഇത് പ്രത്യേകിച്ച് ഉപകാരപ്പെടുന്നു.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## സാരാംശം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

- **പ്രൊവൈഡറെ സൃഷ്ടിക്കുക** `AzureAIProjectAgentProvider` വഴി Azure AI Foundry Agent Service-ലുമായി ബന്ധപ്പെടുന്നതായി.
- ഏജന്റ് നിങ്ങളുടെ Python ഫังก്ഷനുകൾ വിളിക്കാൻ സാധിച്ച് ഒരു **ടൂൾ നിർവചിക്കുക** `@tool` ഡെക്കോറേറ്റർ ഉപയോഗിച്ച്.
- ഉപയോക്തൃ സന്ദേശത്തോടെ **ഏജന്റ് പ്രവർത്തിപ്പിക്കുക** കൂടാതെ അതിന്റെ പ്രതികരണം മুদ্রിക്കുക.
- യഥാർത്ഥ സമയത്തിന്‍റെ ഔട്ട്പുട്ടിന് **പ്രതികരണങ്ങൾ സ്ട്രീം ചെയ്യുക**.

അടുത്ത പാഠത്തിൽ നാം ഏജന്റിക് ഫ്രെയിംവർക്ക്‌സിനെ കൂടുതൽ ആഴത്തിൽ അന്വേഷിച്ച് ഏജന്റുകൾക്ക് ശക്തമായ ടൂളുകളും ബഹുചുവടുള്ള കാരണം പറയലും നൽകുന്നതിനെക്കുറിച്ച് പഠിക്കും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
